# Markov Chain Model

## Intro
In their 2014 paper: Intensity models and transition probabilities for credit card loan delinquencies, Mindy Leow and Jonathan Crook outline an interesting method for predicting credit card defaulting. By creating a non-homogenous Markov Chain between states of credit. Up to date, 1 month in arrears 2 month in arrears and defaulting. A similar methodology could be applied to our dataset. 

## Adaptation
There are three types covariants which could be focused on for this model:
- PAY_n (which tracks repayment status month by month)
- PAY_AMTn (which tracks amount repayed month by month)
- BILL_AMTn (which tracks amount owed month by month)
 
 The statuses can be split into 4 brackets similarly to the paper and then a transition matrix for going from one state to another can be made using a the training data. There are two other time based data which could also be used as foundation for a model.

In [30]:
taiwan = taiwan.copy()
taiwan = taiwan.rename(columns={"PAY_0": "PAY_1"})   # rename for consistency with other columns

#order data chronologically by month (PAY_1 is the most recent month, PAY_6 is the oldest)
months = ["Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct"]
pay_cols  = ["PAY_6", "PAY_5", "PAY_4", "PAY_3", "PAY_2", "PAY_1"]
bill_cols = [f"BILL_AMT{i}" for i in range(6, 0, -1)]
amt_cols  = [f"PAY_AMT{i}" for i in range(6, 0, -1)]

# create table which shows the relevant data from the cleaned dataframe for a given client (row)
def client_table(taiwan, n=0):
    r = taiwan.iloc[n]
    return pd.DataFrame(
        [list(range(1, 8)),
         list(r[pay_cols]) + [r["default payment next month"]],
         list(r[bill_cols]) + [r["default payment next month"]],
         list(r[amt_cols]) + [r["default payment next month"]]],
        index=["Month no.", "Status (PAY / default label)", "BILL_AMT", "PAY_AMT"],
        columns=months,
    )

print(client_table(taiwan, 1))

                               Apr   May   Jun   Jul   Aug   Sep  Oct
Month no.                        1     2     3     4     5     6    7
Status (PAY / default label)     2     0     0     0     2    -1    1
BILL_AMT                      3261  3455  3272  2682  1725  2682    1
PAY_AMT                       2000     0  1000  1000  1000     0    1


## Model 1
A homogeneous markov chain will be simpler to construct and be relatively appropriate since the timeframe we are working with is so short. 

### Splitting Data

The status has values from -2 to 8 Which will be split into categories of lateness:
- If the number is under 0 then it has different meanings: -2 says no payments are needed to be made, -1 payed off in full, and 0 is at least the minimum was payed.
- One month late
- Two months late 
- Over three months late

In [41]:
states = ["current", "1_month_late", "2_months_late", "3plus_months_late"]
def to_state(status):
    if status <= 0:
        return "current"
    if status == 1:
        return "1_month_late"
    if status == 2:
        return "2_months_late"
    return "3plus_months_late"

payment_states = taiwan[pay_cols].apply(lambda col: col.map(to_state))

y = taiwan["default payment next month"].astype(int)

## Creating The Transition Matrix 

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split


train_idx, test_idx = train_test_split(
    np.arange(len(taiwan)), test_size=0.25, random_state=42, stratify=y
)

# Estimate the transition matrix using training clients only.
transition_counts = pd.DataFrame(1.0, index=states, columns=states)  # Laplace smoothing
for idx in train_idx:
    sequence = payment_states.iloc[idx].tolist()
    for source, destination in zip(sequence[:-1], sequence[1:]):
        transition_counts.loc[source, destination] += 1

transition_matrix = transition_counts.div(transition_counts.sum(axis=1), axis=0)

# Smoothed default rate for each observed payment state, from training data.
train_states = payment_states.iloc[train_idx]["PAY_1"]
global_default_rate = y.iloc[train_idx].mean()
state_default_rate = {}

for state in states:
    mask = train_states.eq(state)
    # 20 prior observations shrink estimates for rare states toward the overall rate.
    state_default_rate[state] = (
        y.iloc[train_idx][mask].sum() + 20 * global_default_rate
    ) / (mask.sum() + 20)

# Project one Markov transition from each client's latest payment state.
test_current_states = payment_states.iloc[test_idx]["PAY_1"]
risk_by_state = pd.Series(state_default_rate)
predicted_default_probability = (
    transition_matrix.loc[test_current_states].to_numpy() @ risk_by_state[states].to_numpy()
)

y_test = y.iloc[test_idx].to_numpy()
print("Transition matrix:")
display(transition_matrix)
print(f"ROC AUC: {roc_auc_score(y_test, predicted_default_probability):.3f}")
print(f"Accuracy (0.5 threshold): {accuracy_score(y_test, predicted_default_probability >= 0.5):.3f}")
print("Confusion matrix:")
print(confusion_matrix(y_test, predicted_default_probability >= 0.5))

ValueError: Location based indexing can only have [integer, integer slice (START point is INCLUDED, END point is EXCLUDED), listlike of integers, boolean array] types